# AI vs Human Writing Detection Project

## 1. Background
Natural Language Processing (NLP) focuses on enabling computers to understand and process human language. One important area of NLP is text classification. In this project, we classify whether a piece of text was written by a human or generated by an AI model. This is important because large language models are widely accessible and raise concerns in education, academic integrity, and online content authenticity.

## 2. Topic / Problem
The goal of this project is to classify text as human-written or AI-generated. We design and evaluate both classical machine learning and transformer-based deep learning approaches for this binary classification task.

## 3. Dataset
We use the **ai-and-human-text-dataset** from Kaggle:
- Source: https://www.kaggle.com/datasets/hasanyiitakbulut/ai-and-human-text-dataset
- Input example: text snippets
- Output label: `0 = human`, `1 = AI`
- Size: approximately 6,069 samples

This notebook assumes the dataset CSV is in `data/` under the project root.


In [1]:
# If dependencies are missing, install them once in your environment:
# %pip install pandas numpy scikit-learn nltk torch transformers

import os
import re
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [2]:
# 4. Methods: Data Loading + Preprocessing

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'

# Common filename patterns for this dataset across downloads/shares.
CANDIDATE_PATTERNS = [
    '*ai*human*.csv',
    '*AI*Human*.csv',
    '*text*.csv',
    '*.csv',
]


def find_dataset_csv(data_dir: Path) -> Path:
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Expected data directory at {data_dir}. Create it and place the Kaggle CSV inside."
        )

    candidates = []
    for pattern in CANDIDATE_PATTERNS:
        candidates.extend(sorted(data_dir.glob(pattern)))

    # Keep order, remove duplicates
    seen = set()
    unique = []
    for path in candidates:
        p = str(path.resolve())
        if p not in seen:
            seen.add(p)
            unique.append(path)

    if not unique:
        raise FileNotFoundError(
            "No CSV files found in data/. Download from Kaggle and place the CSV in data/."
        )

    # Prefer a file that looks like AI-vs-human text data.
    for path in unique:
        name = path.name.lower()
        if 'ai' in name and 'human' in name:
            return path

    return unique[0]


def normalize_columns(df: pd.DataFrame):
    # Detect text column
    text_candidates = ['text', 'Text', 'content', 'sentence']
    text_col = next((c for c in text_candidates if c in df.columns), None)

    # Detect label column
    label_candidates = ['generated', 'label', 'Category', 'category', 'target', 'Author', 'author']
    label_col = next((c for c in label_candidates if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(
            f"Could not detect text/label columns. Found columns: {list(df.columns)}"
        )

    out = df[[text_col, label_col]].copy()
    out = out.rename(columns={text_col: 'text', label_col: 'label'})

    # Normalize labels to {0,1} for both numeric and string-backed columns.
    numeric_label = pd.to_numeric(out['label'], errors='coerce')
    if numeric_label.notna().all():
        out['label'] = numeric_label.astype(int)
    else:
        label_map = {
            'human': 0,
            'ai': 1,
            'ai-generated': 1,
            'generated': 1,
            'machine': 1,
            '0': 0,
            '1': 1,
        }
        raw_label = out['label'].astype(str).str.strip().str.lower()
        out['label'] = raw_label.map(label_map)

        if out['label'].isna().any():
            # Fallback for unknown binary class names.
            cats = sorted(raw_label.dropna().unique().tolist())
            if len(cats) == 2:
                if 'human' in cats:
                    other = cats[0] if cats[1] == 'human' else cats[1]
                    out['label'] = raw_label.map({'human': 0, other: 1})
                else:
                    out['label'] = raw_label.map({cats[0]: 0, cats[1]: 1})

    out = out.dropna(subset=['text', 'label']).copy()
    out['label'] = out['label'].astype(int)
    out = out[out['label'].isin([0, 1])]
    out['text'] = out['text'].astype(str)

    return out


def preprocess_text(text: str, stop_words_set: set) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    cleaned = [tok for tok in tokens if tok not in stop_words_set and len(tok) > 1]
    return ' '.join(cleaned)


csv_path = find_dataset_csv(DATA_DIR)
print(f'Using dataset file: {csv_path}')

raw_df = pd.read_csv(csv_path)
df = normalize_columns(raw_df)
print(f'Dataset size after cleanup: {len(df)}')
print('Label distribution:')
print(df['label'].value_counts(normalize=True).sort_index())

stop_words_set = set(stopwords.words('english'))
df['cleaned_text'] = df['text'].apply(lambda x: preprocess_text(x, stop_words_set))


Using dataset file: /Users/darielgutierrez/Desktop/classes/561-final/data/data_for_preprocessing.csv
Dataset size after cleanup: 6069
Label distribution:
label
0    0.494315
1    0.505685
Name: proportion, dtype: float64


In [3]:
# Split once for fair comparison across models
X_train_clean, X_test_clean, y_train, y_test, X_train_raw, X_test_raw = train_test_split(
    df['cleaned_text'],
    df['label'],
    df['text'],
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)

print('Train size:', len(X_train_clean), 'Test size:', len(X_test_clean))
print('Train label distribution:')
print(y_train.value_counts(normalize=True).sort_index())


Train size: 4855 Test size: 1214
Train label distribution:
label
0    0.494336
1    0.505664
Name: proportion, dtype: float64


## 4.1 Baseline Model: TF-IDF + Logistic Regression
We convert preprocessed text to TF-IDF vectors, then train a logistic regression classifier as the baseline machine learning model.


In [4]:
def train_tfidf_baseline(X_train, X_test, y_train, y_test):
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=2000, random_state=SEED)
    model.fit(X_train_tfidf, y_train)

    pred = model.predict(X_test_tfidf)
    metrics = {
        'model': 'TF-IDF + LogisticRegression',
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }

    return model, vectorizer, pred, metrics


baseline_model, tfidf_vectorizer, baseline_pred, baseline_metrics = train_tfidf_baseline(
    X_train_clean, X_test_clean, y_train, y_test
)

print('Baseline classification report:')
print(classification_report(y_test, baseline_pred, digits=4))
pd.DataFrame([baseline_metrics])


Baseline classification report:
              precision    recall  f1-score   support

           0     0.9883    0.9883    0.9883       600
           1     0.9886    0.9886    0.9886       614

    accuracy                         0.9885      1214
   macro avg     0.9885    0.9885    0.9885      1214
weighted avg     0.9885    0.9885    0.9885      1214



,model,accuracy,precision,recall,f1
0,TF-IDF + LogisticRegression,0.988468,0.988599,0.988599,0.988599


## 4.2 Advanced Model: BERT Fine-Tuning (Practical Subset)
For runtime feasibility in local environments, we fine-tune BERT on a controlled subset of the training data.

If `transformers` or `torch` are not available, this section will fail with a clear message. You can install dependencies and rerun.


In [5]:
import torch

try:
    from transformers import (
        BertTokenizerFast,
        BertForSequenceClassification,
        Trainer,
        TrainingArguments,
    )
except Exception as e:
    raise ImportError(
        "Transformers dependencies missing. Install with: %pip install torch transformers"
    ) from e


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
    }


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


BERT_TRAIN_SAMPLES = min(800, len(X_train_raw))
BERT_TEST_SAMPLES = min(400, len(X_test_raw))
MAX_LEN = 128

train_pool = pd.DataFrame({'text': X_train_raw.values, 'label': y_train.values})
test_pool = pd.DataFrame({'text': X_test_raw.values, 'label': y_test.values})

if BERT_TRAIN_SAMPLES < len(train_pool):
    bert_train_df, _ = train_test_split(
        train_pool,
        train_size=BERT_TRAIN_SAMPLES,
        random_state=SEED,
        stratify=train_pool['label'],
    )
else:
    bert_train_df = train_pool

if BERT_TEST_SAMPLES < len(test_pool):
    bert_test_df, _ = train_test_split(
        test_pool,
        train_size=BERT_TEST_SAMPLES,
        random_state=SEED,
        stratify=test_pool['label'],
    )
else:
    bert_test_df = test_pool

print(f'BERT train subset: {len(bert_train_df)} | test subset: {len(bert_test_df)}')

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

train_enc = tokenizer(
    bert_train_df['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
)

test_enc = tokenizer(
    bert_test_df['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
)

train_dataset = TextDataset(train_enc, bert_train_df['label'].tolist())
test_dataset = TextDataset(test_enc, bert_test_df['label'].tolist())

bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='artifacts/bert_outputs',
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=25,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# Pull the last evaluation metrics recorded during training.
eval_logs = [log for log in trainer.state.log_history if 'eval_accuracy' in log]
last_eval = eval_logs[-1] if eval_logs else {}

bert_metrics = {
    'model': 'BERT (fine-tuned subset)',
    'accuracy': float(last_eval.get('eval_accuracy', np.nan)),
    'precision': float(last_eval.get('eval_precision', np.nan)),
    'recall': float(last_eval.get('eval_recall', np.nan)),
    'f1': float(last_eval.get('eval_f1', np.nan)),
}
pd.DataFrame([bert_metrics])


BERT train subset: 800 | test subset: 400


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.107678,0.059122,0.997500,0.995074,1.000000,0.997531


,model,accuracy,precision,recall,f1
0,BERT (fine-tuned subset),0.9975,0.995074,1.0,0.997531


## 5. Evaluation
We evaluate with **accuracy, precision, recall, and F1-score** and compare baseline vs advanced model performance.


In [6]:
results = [baseline_metrics]
if 'bert_metrics' in globals():
    results.append(bert_metrics)

results_df = pd.DataFrame(results)
results_df


,model,accuracy,precision,recall,f1
0,TF-IDF + LogisticRegression,0.988468,0.988599,0.988599,0.988599
1,BERT (fine-tuned subset),0.997500,0.995074,1.000000,0.997531


In [7]:
# Optional: single-text inference helper using the trained BERT model
# Run this only if the BERT section above was executed successfully.

import torch.nn.functional as F

def detect_ai_text(text: str):
    if 'bert_model' not in globals() or 'tokenizer' not in globals():
        raise RuntimeError('BERT model/tokenizer not available. Run the BERT training section first.')

    bert_model.eval()
    inputs = tokenizer(text, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
    device = bert_model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    pred = int(np.argmax(probs))
    confidence = float(probs[pred])
    label_map = {0: 'Human', 1: 'AI-Generated'}

    return {
        'prediction': label_map[pred],
        'confidence': round(confidence, 4),
        'prob_human': round(float(probs[0]), 4),
        'prob_ai': round(float(probs[1]), 4),
    }


sample_text = 'I wrote this paragraph after reading three research papers and summarizing their findings.'
print(detect_ai_text(sample_text))


{'prediction': 'AI-Generated', 'confidence': 0.7852, 'prob_human': 0.2148, 'prob_ai': 0.7852}


## 6. BERT vs OpenAI (Prompted LLM) Comparison
This section compares predictions from:
- The fine-tuned BERT model in this notebook
- An OpenAI API call configured with different system prompts

Add your OpenAI key to a local `.env` file:
- `OPENAI_API_KEY=your_key_here`
- Optional: `OPENAI_MODEL=gpt-4.1-mini`


In [ ]:
# OpenAI setup and LLM classifier
# If needed: %pip install openai python-dotenv requests

import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')

if not OPENAI_API_KEY:
    raise ValueError('Missing OPENAI_API_KEY. Add it to your .env file.')

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPTS = {
    'strict_forensic': (
        'You are a strict AI-text forensic analyst. Prefer high precision. '
        'Only classify as AI when evidence is strong.'
    ),
    'style_sensitive': (
        'You are a writing-style analyst. Focus on repetition, overly smooth transitions, '
        'and generic abstraction often found in AI writing.'
    ),
    'balanced': (
        'You are a balanced classifier for human vs AI writing. Be concise, calibrated, '
        'and avoid overconfident claims.'
    ),
}


def parse_llm_json(raw_text: str):
    raw_text = raw_text.strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start = raw_text.find('{')
        end = raw_text.rfind('}')
        if start != -1 and end != -1 and end > start:
            return json.loads(raw_text[start:end + 1])
        raise


def openai_detect_with_prompt(text: str, system_prompt: str, model: str = OPENAI_MODEL):
    user_prompt = (
        'Classify the following text as either Human or AI. '
        'Return ONLY valid JSON with keys: label, confidence, rationale. '
        'label must be exactly Human or AI. confidence must be a number in [0,1].\n\n'
        f'Text:\n{text}'
    )

    raw_output = ''
    try:
        response = client.responses.create(
            model=model,
            input=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            temperature=0,
            max_output_tokens=220,
        )
        raw_output = response.output_text
    except Exception:
        # Fallback for SDK/model combos where chat.completions is more reliable.
        response = client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            temperature=0,
            max_tokens=220,
        )
        raw_output = response.choices[0].message.content

    parsed = parse_llm_json(raw_output)
    label = str(parsed.get('label', '')).strip()
    conf = float(parsed.get('confidence', 0.0))
    rationale = str(parsed.get('rationale', '')).strip()

    if label not in {'Human', 'AI'}:
        raise ValueError(f'Invalid label from OpenAI output: {label}')

    return {
        'label': label,
        'confidence': max(0.0, min(1.0, conf)),
        'rationale': rationale,
        'raw': raw_output,
    }


def bert_detect_for_compare(text: str):
    bert_out = detect_ai_text(text)
    label = 'AI' if bert_out['prediction'] == 'AI-Generated' else 'Human'
    confidence = float(bert_out['confidence']) / 100.0
    return {'label': label, 'confidence': confidence}


In [ ]:
# Build test set: online text + explicit AI-generated text

def fetch_wikipedia_summary(title: str):
    url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{title}'
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    payload = r.json()
    return payload.get('extract', ''), url

samples = []

online_titles = [
    'Natural_language_processing',
    'Academic_integrity',
    'Machine_learning',
]

for title in online_titles:
    try:
        text, src = fetch_wikipedia_summary(title)
        if text:
            samples.append({
                'sample_id': f'online_{title.lower()}',
                'source': src,
                'kind': 'online_human_reference',
                'text': text,
            })
    except Exception as exc:
        print(f'Could not fetch {title}: {exc}')

# Explicit AI-generated examples
samples.append({
    'sample_id': 'ai_generated_1',
    'source': 'synthetic_ai_example',
    'kind': 'ai_generated',
    'text': (
        'In todays rapidly evolving digital ecosystem, leveraging interdisciplinary synergy '        'is essential for scalable innovation. Organizations that align stakeholder narratives '        'with adaptive frameworks can unlock transformative value across complex systems.'
    ),
})
samples.append({
    'sample_id': 'ai_generated_2',
    'source': 'synthetic_ai_example',
    'kind': 'ai_generated',
    'text': (
        'The seminar explores computational linguistics through a multi-layered lens, '        'integrating probabilistic reasoning, semantic abstraction, and iterative refinement '        'to generate coherent discourse in constrained settings.'
    ),
})

samples_df = pd.DataFrame(samples)
samples_df[['sample_id', 'kind', 'source']]


In [ ]:
# Run BERT vs OpenAI under multiple system prompts
comparison_rows = []

for row in samples:
    text = row['text']

    bert_result = bert_detect_for_compare(text)
    comparison_rows.append({
        'sample_id': row['sample_id'],
        'source': row['source'],
        'kind': row['kind'],
        'model': 'BERT',
        'prompt_name': 'n/a',
        'prediction': bert_result['label'],
        'confidence': round(bert_result['confidence'], 4),
        'rationale': '',
    })

    for prompt_name, system_prompt in SYSTEM_PROMPTS.items():
        llm_result = openai_detect_with_prompt(text=text, system_prompt=system_prompt)
        comparison_rows.append({
            'sample_id': row['sample_id'],
            'source': row['source'],
            'kind': row['kind'],
            'model': f'OpenAI ({OPENAI_MODEL})',
            'prompt_name': prompt_name,
            'prediction': llm_result['label'],
            'confidence': round(llm_result['confidence'], 4),
            'rationale': llm_result['rationale'],
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
# Compact view: only predictions/confidence for quick comparison
comparison_df[['sample_id', 'kind', 'model', 'prompt_name', 'prediction', 'confidence']].sort_values(
    ['sample_id', 'model', 'prompt_name']
)


## Conclusion
- The TF-IDF baseline provides a strong and efficient reference.
- BERT can capture richer context and may improve classification quality, but requires more compute.
- For deployment, choose the model based on the tradeoff between performance and runtime/cost constraints.
